# Capstone: Imagine Your Way to the Goal

Capstone notebook for the [World Models & Imagination](https://josephausterweil.github.io/probintro/deep/world-models-imagination/) chapter (Part VIII).

**The experiment.** The chapter fit a next-state predictor on a 1-D corridor and planned inside it. Here you run the full idea one dimension up, on a **gridworld**, and put model-based planning head-to-head with model-free Q-learning on the currency that matters: **how many *real* steps does each need to solve the maze?** Imagined steps are free — that is the model-based agent's whole edge, and its whole risk.

- **Part A — fit a dynamics model.** Collect transitions from a random policy and learn $P(s' \mid s, a)$ with a small network (the corridor's next-state predictor, one dimension up).
- **Part B — plan by imagination.** Random-shooting: sample action sequences, roll the *learned* model forward, keep the best, execute it. Compare its **real-step** cost to the model-free baseline.
- **Part C (stretch) — break it on purpose.** Inject model error and watch planning degrade as the horizon grows — the compounding-error lesson, made quantitative.

The setup, gridworld, data generator, **and a working model-free Q-learning baseline** are complete and runnable. The model fit, the planner, and the error injection are `TODO`s — each stub states its recipe. Work top to bottom.

*Anchors: Dyna (Sutton, 1991) and MuZero (Schrittwieser et al., 2020), honestly described in the chapter.*

## 0. Setup (complete — run me)

In [ ]:
import numpy as np
import jax
import jax.numpy as jnp
import jax.random as jr
from jax import grad, vmap

key = jr.PRNGKey(0)
print('JAX ready:', jax.__version__)

## 1. The gridworld + a random-policy data generator (complete — run me)

A $5\times5$ grid. The trainee starts at the top-left corner (state `START`) and wants the bottom-right (state `GOAL`). Four actions — up, down, left, right — move deterministically and are clipped by the walls. Reaching `GOAL` gives reward $1$ and resets the episode. The learner never sees `step`; it only sees the transitions it collects.

In [ ]:
GRID = 5
S = GRID * GRID              # 25 states, indexed s = row * GRID + col
A = 4                        # 0 up, 1 down, 2 left, 3 right
START, GOAL = 0, S - 1       # (0,0) -> (4,4)
MOVES = [(-1, 0), (1, 0), (0, -1), (0, 1)]

def to_rc(s):   return (s // GRID, s % GRID)
def to_s(r, c): return r * GRID + c

def step(s, a):                              # the REAL dynamics -- hidden from the learner
    r, c = to_rc(s)
    dr, dc = MOVES[a]
    r = min(max(r + dr, 0), GRID - 1)
    c = min(max(c + dc, 0), GRID - 1)
    return to_s(r, c)

def reward(s):  return 1.0 if s == GOAL else 0.0

def collect_random(seed=0, n_steps=4000):    # wander at random; record (s, a, s')
    rng = np.random.default_rng(seed)
    Sd, Ad, S2d = [], [], []
    s = START
    for _ in range(n_steps):
        a = int(rng.integers(A))
        s2 = step(s, a)
        Sd.append(s); Ad.append(a); S2d.append(s2)
        s = START if s2 == GOAL else s2
    return np.array(Sd), np.array(Ad), np.array(S2d)

s_data, a_data, s2_data = collect_random(0)
print("collected", len(s_data), "transitions on a", GRID, "x", GRID, "grid")
print("start", to_rc(START), " goal", to_rc(GOAL))

## 2. The model-free baseline: tabular Q-learning (complete — run me)

Your opponent in the race. Plain Q-learning from the [Q-learning chapter](https://josephausterweil.github.io/probintro/decisions/q-learning/) — no model, just cached values — reporting the number of **real environment steps** it takes before its greedy policy can reach the goal. That real-step count is the number Part B tries to beat.

In [ ]:
def argmax_rand(rng, v):                      # argmax with RANDOM tie-breaking (real exploration)
    return int(rng.choice(np.flatnonzero(v == v.max())))

def greedy_reaches_goal(Q, cap=50):           # DETERMINISTIC greedy; an untrained Q stays stuck
    s, seen = START, set()
    for _ in range(cap):
        if s == GOAL: return True
        a = int(np.argmax(Q[s]))
        if (s, a) in seen: return False       # greedy policy is cycling -> not solved yet
        seen.add((s, a)); s = step(s, a)
    return s == GOAL

def q_learning_baseline(seed=0, alpha=0.5, gamma=0.95, eps=0.1, max_real_steps=40000):
    rng = np.random.default_rng(seed)
    Q = np.zeros((S, A))
    s = START
    for t in range(1, max_real_steps + 1):
        a = int(rng.integers(A)) if rng.random() < eps else argmax_rand(rng, Q[s])
        s2 = step(s, a)
        target = reward(s2) + gamma * (0.0 if s2 == GOAL else Q[s2].max())
        Q[s, a] += alpha * (target - Q[s, a])
        s = START if s2 == GOAL else s2
        if t % 200 == 0 and greedy_reaches_goal(Q):
            return Q, t
    return Q, max_real_steps

Qmf, mf_steps = q_learning_baseline()
print("model-free Q-learning solved the grid in", mf_steps, "REAL steps")

## 3. Part A — fit a world model — TODO

Learn the dynamics $P(s' \mid s, a)$ from the collected transitions. Because next-states are discrete cells, the clean choice is a small **classifier**: one output score per possible next cell, trained with cross-entropy (the categorical negative log-likelihood — the classification twin of the chapter's squared-error Gaussian NLL). Reuse the [From Rules to Weights](https://josephausterweil.github.io/probintro/deep/from-rules-to-weights/) training pattern: `jax.grad` + a gradient-descent loop.

In [ ]:
def one_hot(i, n):
    return jnp.eye(n)[i]

def fit_world_model(s_data, a_data, s2_data, key=jr.PRNGKey(0), H=64, steps=3000, lr=0.1):
    """Return predict(s, a) -> next state (int), a learned model of the dynamics.

    Suggested design (a next-cell classifier):
      x       = concat(one_hot(s, S), one_hot(a, A))     # input, length S + A
      logits  = W2 @ relu(W1 @ x + b1) + b2              # length S, a score per next cell
      loss    = mean cross-entropy(logits, true s')      # = categorical NLL
      predict(s, a) = argmax logits                      # most likely next cell

    TODO:
      1. build X (n, S+A) from s_data, a_data via one_hot; targets = s2_data
      2. init params {W1:(H,S+A), b1:(H), W2:(S,H), b2:(S)} from `key`
      3. loss(params) = mean softmax-cross-entropy of logits vs targets
      4. gradient-descent loop with jax.grad (see the chapter's 2-4-1 cell)
      5. return lambda s, a: int(argmax logits(params, s, a))
    """
    raise NotImplementedError

# When it works, accuracy on the collected data should be high (~0.9+):
# predict = fit_world_model(s_data, a_data, s2_data)
# acc = np.mean([predict(int(s), int(a)) == int(s2)
#                for s, a, s2 in zip(s_data, a_data, s2_data)])
# print("world-model next-state accuracy:", round(float(acc), 3))

## 4. Part B — plan by imagination, then race — TODO

**Random shooting**, the simplest planner that works: sample many random action sequences, roll the *learned* model forward from `START` for each (all imagined — zero real steps), keep the sequence whose imagined rollout reaches `GOAL` soonest, and execute it once for real.

Then answer the race honestly. The model-based agent's **real-step** cost is only the transitions it spent *collecting data to fit the model* — everything after is imagined and free. The model-free `mf_steps` is all real. But note the honest catch this toy exposes: a $5\times5$ grid is so small that Q-learning already solves it in a few hundred real steps, so fitting a model on the default $4000$ transitions is *overkill*. The real question is therefore: **how far can you shrink the data budget** (`collect_random(0, n_steps=…)`) and still fit a model good enough to plan a winning route? That smallest budget is your true real-step cost — find where it crosses `mf_steps`, and you have measured exactly when model-based planning pays for itself.

In [ ]:
def plan_by_imagination(predict, n_plans=400, horizon=16, seed=0):
    """Random-shooting planner INSIDE the learned model.

    TODO:
      1. rng = np.random.default_rng(seed)
      2. for each of n_plans: sample a length-`horizon` action sequence
      3. roll `predict` forward from START; record the first step it reaches GOAL (or +inf)
      4. return the action sequence that reached GOAL in the FEWEST imagined steps
    """
    raise NotImplementedError

def executes_to_goal(plan):
    """Run an action sequence in the REAL world; return True if it reaches GOAL."""
    s = START
    for a in plan:
        s = step(s, int(a))
        if s == GOAL: return True
    return False

# When plan_by_imagination works:
# best = plan_by_imagination(predict)
# print("imagined plan reaches the real goal:", executes_to_goal(best))
# print("model-based real steps (= data budget you fit on):", len(s_data))
# print("model-free  real steps (Q-learning):", mf_steps)
# Now shrink the data budget: refit on collect_random(0, n_steps=SMALLER) and
# find the smallest budget whose model still plans a winning route.

## 5. Part C (stretch) — inject model error, watch planning degrade — TODO

The chapter's central warning: **model error compounds**. Wrap the learned `predict` so a fraction of its queries return a *wrong* next cell, then re-plan at growing horizons. A short plan tolerates a sloppy model; a long one feeds each error back into the next step until the imagined rollout is fiction. Plot "reached goal?" (or imagined-vs-real endpoint distance) against horizon and against the corruption rate.

In [ ]:
def corrupt(predict, p_flip=0.15, seed=0):
    """Return a noisy model: with prob p_flip, predict a random wrong next cell.

    TODO:
      1. rng = np.random.default_rng(seed)
      2. return a function noisy(s, a) that calls predict(s, a) but, with prob p_flip,
         returns a random OTHER state instead
    """
    raise NotImplementedError

# Sweep the horizon with a corrupted model and watch success fall off:
# for h in [4, 8, 12, 16, 20, 24]:
#     best = plan_by_imagination(corrupt(predict, p_flip=0.15), horizon=h)
#     print("horizon", h, "-> reaches real goal:", executes_to_goal(best))
#
# Then sweep p_flip at a fixed long horizon to see the same lesson from the other axis.

---

*This project is generously funded by the Japanese Probabilistic Computing Consortium Association ([JPCCA](https://jpcca.org/)).*